# Solar Modulation Effects on Cosmic Ray Flux

Galactic cosmic rays are decelerated by the solar wind before reaching Earth. GSF uses the force-field approximation, parameterized by a single potential $\phi$ (GV): higher $\phi$ → stronger modulation, lower observed flux. Typical values: $\phi \approx 0.4$ GV at solar minimum, $\phi \approx 1.0$ GV at solar maximum. The unmodulated Local Interstellar Spectrum (LIS) corresponds to $\phi = 0$.

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize

from globalsplinefit import GSFEnergy, GSFEnergyPerNucleon, GSFRigidity

plt.rcParams["figure.figsize"] = (10, 6)

## Three model types, three solar conditions

We compare three GSF model classes against three solar conditions: LIS, solar minimum (2009), solar maximum (1991).

In [ ]:
grid = np.logspace(np.log10(0.5), np.log10(200), 200)  # GeV or GV
energy = rigidity = energy_per_nucleon = grid

gsf_energy = GSFEnergy()
gsf_rigidity = GSFRigidity()
gsf_nucleon = GSFEnergyPerNucleon()

time_periods = {
    "LIS": "LIS",
    "Solar Minimum (2009)": (200901, 200912),
    "Solar Maximum (1991)": (199101, 199112),
}
groups = ["p", "He", "O*", "Fe*"]
group_colors = ["red", "orange", "green", "blue"]
line_styles = {"LIS": "-", "Solar Minimum (2009)": "--", "Solar Maximum (1991)": ":"}
energy_exponent = 2.6

## Flux comparisons

Compute and plot per-group + total flux for each model and time period. Fluxes are scaled by $E^{2.6}$ to flatten the spectrum.

In [ ]:
# Collect fluxes for each (model, period, group) combination
models = {
    "GSFEnergy": (gsf_energy, energy),
    "GSFRigidity": (gsf_rigidity, rigidity),
    "GSFEnergyPerNucleon": (gsf_nucleon, energy_per_nucleon),
}

fluxes = {}  # fluxes[model_name][period][group_or_"Total"]
for mname, (model, grid_) in models.items():
    fluxes[mname] = {}
    for period, ti in time_periods.items():
        fluxes[mname][period] = {
            g: model.flux(grid_, g, time_interval=ti) for g in groups
        }
        fluxes[mname][period]["Total"] = model.total_flux(grid_, time_interval=ti)

In [ ]:
def plot_groups(ax, model_name, grid_, xlabel, scale_label):
    period_handles = []
    for i, (group, color) in enumerate(zip(groups, group_colors, strict=False)):
        for period, style in line_styles.items():
            flux = fluxes[model_name][period][group]
            (line,) = ax.loglog(
                grid_, flux * grid_**energy_exponent, ls=style, color=color, alpha=0.8
            )
            if i == 0:
                period_handles.append((line, period))
    group_handles = [
        ax.plot([], [], color=c, lw=3, label=lbl)[0]
        for c, lbl in zip(
            group_colors, ["Proton", "Helium", "Oxygen*", "Iron*"], strict=False
        )
    ]
    ax.set_xlabel(xlabel)
    ax.set_ylabel(scale_label)
    ax.set_title(model_name)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(grid_[0], grid_[-1])
    period_legend = ax.legend(
        [h for h, _ in period_handles],
        [lbl for _, lbl in period_handles],
        loc="lower right",
    )
    ax.legend(handles=group_handles, loc="upper right", ncol=4)
    ax.add_artist(period_legend)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_groups(
    axes[0], "GSFEnergy", energy, "Energy [GeV]", f"Flux × $E^{{{energy_exponent}}}$"
)
plot_groups(
    axes[1],
    "GSFRigidity",
    rigidity,
    "Rigidity [GV]",
    f"Flux × $R^{{{energy_exponent}}}$",
)
plot_groups(
    axes[2],
    "GSFEnergyPerNucleon",
    energy_per_nucleon,
    "Energy/nucleon [GeV/nucleon]",
    f"Nucleon flux × $E^{{{energy_exponent}}}$",
)
for ax in axes:
    ax.set_ylim(1e0, 3e4)
plt.tight_layout()
plt.show()

### Total flux across model types

Overlay the total flux from all three models on a single axis. Note the x-axis units differ by model (GeV vs GV vs GeV/nucleon), so this exposes structural differences rather than direct comparisons.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_models = [
    ("GSFEnergy", energy, "black"),
    ("GSFRigidity", rigidity, "purple"),
    ("GSFEnergyPerNucleon", energy_per_nucleon, "brown"),
]
for mname, grid_, color in plot_models:
    for period, style in line_styles.items():
        flux = fluxes[mname][period]["Total"]
        label = mname if period == "LIS" else None
        ax.loglog(
            grid_,
            flux * grid_**energy_exponent,
            ls=style,
            color=color,
            label=label,
            alpha=0.8,
        )

ax.set_xlabel("Energy / Rigidity / Energy per nucleon")
ax.set_ylabel(f"Total flux × $X^{{{energy_exponent}}}$ (units vary by model)")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right")
ax.set_ylim(5e2, 3e4)
plt.tight_layout()
plt.show()

## Solar cycle evolution

Sample 17 monthly time points across 2013–2024 and compute the total nucleon flux at each. The color scale maps to $\phi$ (warmer = stronger modulation). The second figure shows $\phi$ and the flux-suppression ratio (modulated / LIS) at 1.5, 10, and 100 GeV/nucleon.

In [ ]:
# Sample 17 monthly time points between 2013-01 and 2024-12
total_months = (2024 - 2013) * 12 + 12
step = total_months // 16
time_points = []
phi_values = []
dates = []
month_idx = 0
for _ in range(17):
    year = 2013 + month_idx // 12
    month = (month_idx % 12) + 1
    if year in gsf_energy.phi:
        time_points.append((year * 100 + month, year * 100 + month + 1))
        phi_values.append(float(gsf_energy.phi[year][month - 1]))
        dates.append(f"{year}-{month:02d}")
    month_idx += step

print(
    f"{len(time_points)} time points, φ range: {min(phi_values):.3f}–{max(phi_values):.3f} GV"
)

In [ ]:
time_series_fluxes = np.array(
    [gsf_nucleon.total_flux(energy_per_nucleon, time_interval=ti) for ti in time_points]
)
flux_lis = gsf_nucleon.total_flux(energy_per_nucleon, time_interval="LIS")

In [ ]:
cmap = cm.get_cmap("coolwarm")
norm = Normalize(vmin=min(phi_values), vmax=max(phi_values))

fig, ax = plt.subplots(figsize=(8, 5))
for flux, phi, _date in zip(time_series_fluxes, phi_values, dates, strict=False):
    ax.loglog(
        energy_per_nucleon,
        flux * energy_per_nucleon**energy_exponent,
        color=cmap(norm(phi)),
        alpha=0.7,
        lw=1.5,
    )
ax.loglog(
    energy_per_nucleon,
    flux_lis * energy_per_nucleon**energy_exponent,
    "k-",
    lw=3,
    label="LIS (φ=0)",
)
ax.set_xlabel("Energy [GeV/nucleon]")
ax.set_ylabel(f"Nucleon flux × $E^{{{energy_exponent}}}$")
ax.set_title("Nucleon flux through the solar cycle")
ax.grid(True, alpha=0.3)
ax.set_xlim(energy_per_nucleon[0], energy_per_nucleon[-1])
ax.set_ylim(5e2, 3e4)
ax.legend(loc="lower right")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label="φ [GV]")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax2 = ax.twinx()

years = [int(d[:4]) + (int(d[5:7]) - 1) / 12 for d in dates]
ax.plot(years, phi_values, "ro-", lw=2, ms=6, label="φ")
ax.set_xlabel("Year")
ax.set_ylabel("φ [GV]", color="red")
ax.tick_params(axis="y", labelcolor="red")
ax.grid(True, alpha=0.3)

for energy_val, color in zip(
    [1.5, 10.0, 100.0], ["blue", "green", "orange"], strict=False
):
    idx = np.argmin(np.abs(energy_per_nucleon - energy_val))
    ratio = time_series_fluxes[:, idx] / flux_lis[idx]
    ax2.plot(
        years,
        ratio,
        "o-",
        color=color,
        lw=2,
        ms=4,
        label=f"{energy_per_nucleon[idx]:.1f} GeV/nuc",
    )

ax2.axhline(1.0, color="gray", ls="--", alpha=0.5)
ax2.set_ylabel("Flux ratio (modulated / LIS)")
ax2.set_ylim(0, 1.2)
ax2.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Approximate vs explicit solar-cycle averaging

`time_interval=None` returns a Solar Cycle 24 average. With `use_approximate_solar_cycle_average=True` (default) the flux is evaluated once at $\langle\phi\rangle$ — fast. With `False` it's averaged over the monthly $\phi$ values — slower but exact. Because flux depends nonlinearly on $\phi$, the two differ by a few percent.

In [ ]:
import time as _time

gsf_approx = GSFEnergy(use_approximate_solar_cycle_average=True)
gsf_exact = GSFEnergy(use_approximate_solar_cycle_average=False)

energy_subset = np.logspace(0, 1, 500)

t0 = _time.time()
flux_approx = gsf_approx.total_flux(energy_subset)
t_approx = _time.time() - t0

t0 = _time.time()
flux_exact = gsf_exact.total_flux(energy_subset)
t_exact = _time.time() - t0

rel_diff = 100 * np.abs(flux_approx - flux_exact) / flux_exact

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(7, 5))
ax1.loglog(
    energy_subset,
    energy_subset**energy_exponent * flux_approx,
    "r-",
    lw=2,
    label=r"Approximate Flux$(\langle\phi\rangle)$",
)
ax1.loglog(
    energy_subset,
    energy_subset**energy_exponent * flux_exact,
    "b--",
    lw=2,
    label=r"Exact $\langle$Flux$(\phi)\rangle$",
)
ax1.set_ylabel("Total flux × $E^{2.6}$")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogx(energy_subset, rel_diff, "g-", lw=2)
ax2.set_xlabel("Energy [GeV]")
ax2.set_ylabel("Relative difference [%]")
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, rel_diff.max() * 1.1)
ax2.text(
    0.7,
    0.95,
    f"Approx: {t_approx:.3f} s\nExact: {t_exact:.3f} s\nSpeedup: {t_exact / t_approx:.1f}×\nMax diff: {rel_diff.max():.2f}%",
    transform=ax2.transAxes,
    va="top",
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.8},
    fontfamily="monospace",
    fontsize=9,
)
plt.tight_layout()
plt.show()